In [5]:
!pip install sqlalchemy pymysql pandas 



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd     # Se importan las librerias a utilizar
import sqlalchemy
import pymysql

In [65]:
from sqlalchemy import create_engine

usuario = "root"          # Datos de coonexion a MySQL
password = "enzoYpelu2026"  
host = "localhost"
puerto = "3306"
base = "tp3_vinos"        # Nombre de la base de datos que voy a usar

engine = create_engine(f"mysql+pymysql://{usuario}:{password}@{host}:{puerto}/{base}")  #motor de conexion a MySQL

engine.connect() # Chequeo de conexion 


In [47]:
df = pd.read_csv("set_datos_arreglo.csv", sep=";", header=None, low_memory=False) # Se lee el set de datos sin la primera fila
                                                            # Porque esta no contiene nombres como column1 y no sera necesaria.
# Se toma la primera fila como encabezado real (con los nombres de las columnas creadas en la tabla de MySQL)
header = df.iloc[1].tolist()

# Me quedo con todas las filas menos las primeras 2 filas (0 y 1)
df = df[2:].reset_index(drop=True)

# Asigno los nombres de columnas correctos
df.columns = header


In [50]:
df = df.iloc[:, :-2]    # Se elimina las dos ultimas columnas NaN que contienen en su gran mayoria datos nulos

In [60]:
cols = df.columns.tolist() 
cols[0] = "num_resena"   # Se renombra la primer columna que estaba como NaN para que coinncida con el nombre en MySQL
df.columns = cols

df = df.rename(columns={"winery;;;;": "winery"}) #Renombro columna winery tenia ; innecesarios

df["winery"] = df["winery"].str.replace(";", "", regex=False)  # Se limpian los ';;;;' del final que contienen las filas en la columna winery


In [64]:
# Chequeo de la estructura del dataframe
df.shape    
df.columns
df.head()

,num_resena,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


In [63]:
# Se craga el dataframe 'limpi' a la tabla loadtable_wines en MySQL
df.to_sql(
    name="loadtable_wines",
    con=engine,
    if_exists="append",       # Se insertan los datos sin borrar la tabla
    index=False,              # No se envía el índice de pandas
    chunksize=5000            # Se carga en bloques para evitar errores por tamaño
)


129971